In [3]:
!pip install cvzone

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for cvzone: filename=cvzone-1.6.1-py3-none-any.whl size=26311 sha256=04c77f0692257257d9bf74081f4a0a95e2c67994ef962cb4b330b6346bbbe4e3
  Stored in directory: c:\users\dell\appdata\local\pip\cache\wheels\5d\21\e8\3147ae88d44e27f06e0175d337a7673c70fb957202cbbe2034
Successfully built cvzone


In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [3]:
#1+2 code
import cv2
import numpy as np
from ultralytics import YOLO
import cvzone
import matplotlib.pyplot as plt
import io

class_names = [
    'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 
    'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 
    'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 
    'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 
    'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 
    'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]

class_names_goal = ['car', 'motorcycle', 'bus', 'truck']

class ResolutionIndependentTrafficMonitor:
    def __init__(self, video_path, model_path=r'C:/Users/DELL/Desktop/only traffic project/yolov8/yolov8x.pt'):
        self.video = cv2.VideoCapture(video_path)
        if not self.video.isOpened():
            raise ValueError("Error opening video file")
        
        self.width = int(self.video.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.height = int(self.video.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.fps = self.video.get(cv2.CAP_PROP_FPS)
        
        # Initialize YOLOv8x model
        self.model = YOLO(model_path)
        
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        output_path = 'C:/Users/DELL/Desktop/only traffic project/vehicle classification/traffic_classify.mp4'
        self.out = cv2.VideoWriter(output_path, fourcc, self.fps, (self.width, self.height))
        
        self.vehicle_count = {'left': 0, 'right': 0}
        self.counted_vehicle_ids = {'left': set(), 'right': set()}
        
        self.line_y = int(self.height * 0.65)
        self.line_x_left = int(self.width * 0.25)
        self.line_x_right = int(self.width * 0.75)
        
        self.mask = self.create_mask()
        self.tracker = SimpleTracker()

    def create_pie_chart(self):
        # Create a figure with a transparent background
        plt.figure(figsize=(4, 4), facecolor='none')
        
        colors = [(219, 0, 115), (0, 255, 127)]
        lanes = list(self.vehicle_count.keys())
        total_counts = [self.vehicle_count[lane] for lane in lanes]
        
        labels = [f'{lane.upper()}: {total_counts[i]}' for i, lane in enumerate(lanes)]
        sizes = total_counts if sum(total_counts) > 0 else [1, 1]
        
        plt.pie(
            sizes, 
            labels=labels, 
            autopct='%1.1f%%', 
            startangle=140, 
            colors=[(r/255, g/255, b/255) for r, g, b in colors],
            wedgeprops={"linewidth": 1, "edgecolor": "white"},
            textprops={"fontsize": 8, "fontweight": "bold", "color": "white"}
        )
        plt.title('Vehicle Distribution', color='white', pad=10, fontsize=10)
        
        # Convert plot to image
        buf = io.BytesIO()
        plt.savefig(buf, format='png', transparent=True, bbox_inches='tight', dpi=100)
        buf.seek(0)
        img_arr = np.frombuffer(buf.getvalue(), dtype=np.uint8)
        buf.close()
        plt.close()
        
        # Convert to BGR for OpenCV
        img = cv2.imdecode(img_arr, cv2.IMREAD_UNCHANGED)
        img = cv2.cvtColor(img, cv2.COLOR_RGBA2BGRA)
        
        return img

    def create_mask(self):
        mask = np.zeros((self.height, self.width), dtype=np.uint8)
        pts = np.array([
            [int(self.width * 0.15), int(self.height * 0.55)], 
            [int(self.width * 0.85), int(self.height * 0.55)], 
            [self.width, self.height], 
            [0, self.height]
        ], np.int32)
        pts = pts.reshape((-1, 1, 2))
        cv2.fillPoly(mask, [pts], 255)
        return mask
    
    def overlay_pie_chart(self, frame, pie_chart):
        # Calculate position for pie chart (top right corner)
        chart_width = int(self.width * 0.3)  # 30% of frame width
        chart_height = int(chart_width * pie_chart.shape[0] / pie_chart.shape[1])
        pie_chart_resized = cv2.resize(pie_chart, (chart_width, chart_height))
        
        # Region of Interest (ROI) in the frame
        roi = frame[10:10+chart_height, self.width-chart_width-10:self.width-10]
        
        # Create a mask from the alpha channel
        alpha_channel = pie_chart_resized[:, :, 3] / 255.0
        alpha_3channel = np.stack([alpha_channel] * 3, axis=-1)
        
        # Blend the images
        blended_roi = (1 - alpha_3channel) * roi + alpha_3channel * pie_chart_resized[:, :, :3]
        frame[10:10+chart_height, self.width-chart_width-10:self.width-10] = blended_roi.astype(np.uint8)
        
        return frame

    def process_frame(self, frame):
        frame = cv2.resize(frame, (self.width, self.height))
        image_region = cv2.bitwise_and(frame, frame, mask=self.mask)
        
        # Detect vehicles
        detections = self.detect_vehicles(image_region)
        
        # Track and count vehicles
        tracked_objects = self.tracker.update(detections)
        self.count_vehicles(frame, tracked_objects)
        
        # Annotate frame with tracking info
        self.annotate_frame(frame, tracked_objects)
        
        # Create and overlay pie chart
        pie_chart = self.create_pie_chart()
        frame = self.overlay_pie_chart(frame, pie_chart)
        
        return frame

    def detect_vehicles(self, image_region):
        detections = []
        results = self.model(image_region, stream=True)
        
        for r in results:
            for box in r.boxes:
                class_name = class_names[int(box.cls[0])]
                if class_name not in class_names_goal:
                    continue
                
                confidence = round(float(box.conf[0]) * 100, 2)
                if confidence < 30:
                    continue
                
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                detections.append([x1, y1, x2, y2, float(box.conf[0]), class_name])
        
        return detections
    
    def count_vehicles(self, frame, tracked_objects):
        for obj in tracked_objects:
            x1, y1, x2, y2, obj_id, cls = obj
            center_x, center_y = (x1 + x2) // 2, (y1 + y2) // 2
            
            if self.line_y - 10 < center_y < self.line_y + 10:
                if center_x < self.width // 2:  # Left side
                    if obj_id not in self.counted_vehicle_ids['left']:
                        self.counted_vehicle_ids['left'].add(obj_id)
                        self.vehicle_count['left'] += 1
                        cv2.line(frame, 
                                (0, self.line_y), 
                                (self.width // 2, self.line_y), 
                                (0, 255, 0), 2)
                else:  # Right side
                    if obj_id not in self.counted_vehicle_ids['right']:
                        self.counted_vehicle_ids['right'].add(obj_id)
                        self.vehicle_count['right'] += 1
                        cv2.line(frame, 
                                (self.width // 2, self.line_y), 
                                (self.width, self.line_y), 
                                (0, 255, 0), 2)
    
    def annotate_frame(self, frame, tracked_objects):
        for obj in tracked_objects:
            x1, y1, x2, y2, obj_id, cls = obj
            confidence_pos_x1 = max(0, x1)
            confidence_pos_y1 = max(36, y1)
            
            cvzone.putTextRect(frame, 
                            f'ID: {obj_id} {cls}', 
                            (confidence_pos_x1, confidence_pos_y1), 
                            1, 1)
        
        cvzone.putTextRect(frame, 
                        f'Left Lane: {self.vehicle_count["left"]}', 
                        (50, 50), 2, 2, 
                        offset=20, border=2, 
                        colorR=(127, 0, 255), colorB=(127, 0, 255))
        
        cvzone.putTextRect(frame, 
                        f'Right Lane: {self.vehicle_count["right"]}', 
                        (self.width - 250, 50), 2, 2, 
                        offset=20, border=2, 
                        colorR=(127, 0, 255), colorB=(127, 0, 255))
    
    def process_video(self):
        while True:
            success, frame = self.video.read()
            if not success:
                break
            
            processed_frame = self.process_frame(frame)
            
            cv2.imshow('Traffic Monitoring', processed_frame)
            self.out.write(processed_frame)
            
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        
        self.video.release()
        self.out.release()
        cv2.destroyAllWindows()

class SimpleTracker:
    def __init__(self, max_age=20):
        self.next_id = 1
        self.tracked_objects = {}
        self.max_age = max_age

    def update(self, detections):
        new_tracked_objects = {}
        
        for det in detections:
            x1, y1, x2, y2, conf, cls = det
            center = ((x1 + x2) // 2, (y1 + y2) // 2)
            
            best_match = None
            min_distance = float('inf')
            
            for obj_id, obj in self.tracked_objects.items():
                dist = np.linalg.norm(np.array(center) - np.array(obj['center']))
                if dist < min_distance:
                    min_distance = dist
                    best_match = obj_id
            
            if best_match is not None and min_distance < 50:
                new_tracked_objects[best_match] = {
                    'bbox': (x1, y1, x2, y2),
                    'center': center,
                    'class': cls,
                    'age': 0
                }
            else:
                new_tracked_objects[self.next_id] = {
                    'bbox': (x1, y1, x2, y2),
                    'center': center,
                    'class': cls,
                    'age': 0
                }
                self.next_id += 1
        
        for obj_id in self.tracked_objects:
            if obj_id not in new_tracked_objects:
                self.tracked_objects[obj_id]['age'] += 1
                if self.tracked_objects[obj_id]['age'] < self.max_age:
                    new_tracked_objects[obj_id] = self.tracked_objects[obj_id]
        
        self.tracked_objects = new_tracked_objects
        return [(obj['bbox'][0], obj['bbox'][1], obj['bbox'][2], obj['bbox'][3], obj_id, obj['class']) 
                for obj_id, obj in self.tracked_objects.items()]

def main():
    video_path = r'C:/Users/DELL/Desktop/only traffic project/vehicle classification/traffic.mp4'
    monitor = ResolutionIndependentTrafficMonitor(video_path)
    monitor.process_video()

if __name__ == "__main__":
    main()


0: 384x640 4 persons, 9 cars, 3 motorcycles, 3 trucks, 28.8ms
Speed: 1.0ms preprocess, 28.8ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 9 cars, 3 motorcycles, 2 trucks, 30.4ms
Speed: 0.0ms preprocess, 30.4ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 9 cars, 3 motorcycles, 2 trucks, 26.0ms
Speed: 0.0ms preprocess, 26.0ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 persons, 9 cars, 3 motorcycles, 3 trucks, 19.4ms
Speed: 1.0ms preprocess, 19.4ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 9 cars, 3 motorcycles, 2 trucks, 18.6ms
Speed: 0.0ms preprocess, 18.6ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 9 cars, 3 motorcycles, 2 trucks, 14.4ms
Speed: 1.0ms preprocess, 14.4ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 9 cars, 3 motorcy

In [7]:
#classify1
import cv2
import numpy as np
from ultralytics import YOLO
import cvzone
import matplotlib.pyplot as plt
import io

class_names = [
    'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 
    'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 
    'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 
    'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 
    'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 
    'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]

class_names_goal = ['car', 'motorcycle', 'bus', 'truck']

class ResolutionIndependentTrafficMonitor:
    def __init__(self, video_path, model_path=r'C:/Users/DELL/Desktop/only traffic project/yolov8/yolov8x.pt'):
        self.video = cv2.VideoCapture(video_path)
        if not self.video.isOpened():
            raise ValueError("Error opening video file")
        
        self.width = int(self.video.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.height = int(self.video.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.fps = self.video.get(cv2.CAP_PROP_FPS)
        
        # Initialize YOLOv8x model
        self.model = YOLO(model_path)
        
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        output_path = 'C:/Users/DELL/Desktop/only traffic project/vehicle classification/traffic_classify2.mp4'
        self.out = cv2.VideoWriter(output_path, fourcc, self.fps, (self.width, self.height))
        
        self.vehicle_count = {'left': 0, 'right': 0}
        self.counted_vehicle_ids = {'left': set(), 'right': set()}
        
        self.line_y = int(self.height * 0.65)
        self.line_x_left = int(self.width * 0.25)
        self.line_x_right = int(self.width * 0.75)
        
        self.mask = self.create_mask()
        self.tracker = SimpleTracker()

    def create_pie_chart(self):
        # Create a figure with a transparent background
        plt.figure(figsize=(4, 4), facecolor='none')
        
        colors = [(219, 0, 115), (0, 255, 127)]
        lanes = list(self.vehicle_count.keys())
        total_counts = [self.vehicle_count[lane] for lane in lanes]
        
        labels = [f'{lane.upper()}: {total_counts[i]}' for i, lane in enumerate(lanes)]
        sizes = total_counts if sum(total_counts) > 0 else [1, 1]
        
        plt.pie(
            sizes, 
            labels=labels, 
            autopct='%1.1f%%', 
            startangle=140, 
            colors=[(r/255, g/255, b/255) for r, g, b in colors],
            wedgeprops={"linewidth": 1, "edgecolor": "white"},
            textprops={"fontsize": 12, "fontweight": "bold", "color": "white"}
        )
        plt.title('Vehicle Distribution', color='white', pad=10, fontsize=14)
        
        # Convert plot to image
        buf = io.BytesIO()
        plt.savefig(buf, format='png', transparent=True, bbox_inches='tight', dpi=100)
        buf.seek(0)
        img_arr = np.frombuffer(buf.getvalue(), dtype=np.uint8)
        buf.close()
        plt.close()
        
        # Convert to BGR for OpenCV
        img = cv2.imdecode(img_arr, cv2.IMREAD_UNCHANGED)
        img = cv2.cvtColor(img, cv2.COLOR_RGBA2BGRA)
        
        return img

    def create_mask(self):
        mask = np.zeros((self.height, self.width), dtype=np.uint8)
        pts = np.array([ 
            [int(self.width * 0.15), int(self.height * 0.55)], 
            [int(self.width * 0.85), int(self.height * 0.55)], 
            [self.width, self.height], 
            [0, self.height] 
        ], np.int32)
        pts = pts.reshape((-1, 1, 2))
        cv2.fillPoly(mask, [pts], 255)
        return mask
    
    def overlay_pie_chart(self, frame, pie_chart):
        # Calculate position for pie chart (right side)
        chart_width = int(self.width * 0.3)  # 30% of frame width
        chart_height = int(chart_width * pie_chart.shape[0] / pie_chart.shape[1])
        pie_chart_resized = cv2.resize(pie_chart, (chart_width, chart_height))
        
        # Region of Interest (ROI) in the frame
        roi = frame[10:10+chart_height, self.width-chart_width-10:self.width-10]
        
        # Create a mask from the alpha channel
        alpha_channel = pie_chart_resized[:, :, 3] / 255.0
        alpha_3channel = np.stack([alpha_channel] * 3, axis=-1)
        
        # Blend the images
        blended_roi = (1 - alpha_3channel) * roi + alpha_3channel * pie_chart_resized[:, :, :3]
        frame[10:10+chart_height, self.width-chart_width-10:self.width-10] = blended_roi.astype(np.uint8)
        
        return frame

    def process_frame(self, frame):
        frame = cv2.resize(frame, (self.width, self.height))
        image_region = cv2.bitwise_and(frame, frame, mask=self.mask)
        
        # Detect vehicles
        detections = self.detect_vehicles(image_region)
        
        # Track and count vehicles
        tracked_objects = self.tracker.update(detections)
        self.count_vehicles(frame, tracked_objects)
        
        # Annotate frame with tracking info
        self.annotate_frame(frame, tracked_objects)
        
        # Create and overlay pie chart
        pie_chart = self.create_pie_chart()
        frame = self.overlay_pie_chart(frame, pie_chart)
        
        return frame

    def detect_vehicles(self, image_region):
        detections = []
        results = self.model(image_region, stream=True)
        
        for r in results:
            for box in r.boxes:
                class_name = class_names[int(box.cls[0])]
                if class_name not in class_names_goal:
                    continue
                
                confidence = round(float(box.conf[0]) * 100, 2)
                if confidence < 60:  # Increased confidence threshold to reduce false positives
                    continue
                
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                detections.append([x1, y1, x2, y2, float(box.conf[0]), class_name])
        
        return detections
    
    def count_vehicles(self, frame, tracked_objects):
        for obj in tracked_objects:
            x1, y1, x2, y2, obj_id, cls = obj
            center_x, center_y = (x1 + x2) // 2, (y1 + y2) // 2
            
            if self.line_y - 10 < center_y < self.line_y + 10:
                if center_x < self.width // 2:  # Left side
                    if obj_id not in self.counted_vehicle_ids['left']:
                        self.counted_vehicle_ids['left'].add(obj_id)
                        self.vehicle_count['left'] += 1
                        cv2.line(frame, 
                                (0, self.line_y), 
                                (self.width // 2, self.line_y), 
                                (0, 255, 0), 2)
                else:  # Right side
                    if obj_id not in self.counted_vehicle_ids['right']:
                        self.counted_vehicle_ids['right'].add(obj_id)
                        self.vehicle_count['right'] += 1
                        cv2.line(frame, 
                                (self.width // 2, self.line_y), 
                                (self.width, self.line_y), 
                                (0, 255, 0), 2)
    
    def annotate_frame(self, frame, tracked_objects):
        for obj in tracked_objects:
            x1, y1, x2, y2, obj_id, cls = obj
            confidence_pos_x1 = max(0, x1)
            confidence_pos_y1 = max(36, y1)
            
            cvzone.putTextRect(frame, 
                            f'{cls}', 
                            (confidence_pos_x1, confidence_pos_y1), 
                            1, 1)
        
        cvzone.putTextRect(frame, 
                        f'Left Lane: {self.vehicle_count["left"]}', 
                        (50, 50), 2, 2, 
                        offset=20, border=2, 
                        colorR=(0, 128, 255), colorB=(0, 128, 255))
        
        cvzone.putTextRect(frame, 
                        f'Right Lane: {self.vehicle_count["right"]}', 
                        (self.width - 250, 50), 2, 2, 
                        offset=20, border=2, 
                        colorR=(0, 128, 255), colorB=(0, 128, 255))
    
    def process_video(self):
        while True:
            success, frame = self.video.read()
            if not success:
                break
            
            processed_frame = self.process_frame(frame)
            
            # Display video and pie chart
            cv2.imshow('Traffic Monitoring', processed_frame)
            self.out.write(processed_frame)
            
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        
        self.video.release()
        self.out.release()
        cv2.destroyAllWindows()

class SimpleTracker:
    def __init__(self, max_age=20):
        self.next_id = 1
        self.tracked_objects = {}
        self.max_age = max_age

    def update(self, detections):
        new_tracked_objects = {}
        
        for det in detections:
            x1, y1, x2, y2, conf, cls = det
            center = ((x1 + x2) // 2, (y1 + y2) // 2)
            
            best_match = None
            min_distance = float('inf')
            
            for obj_id, obj in self.tracked_objects.items():
                dist = np.linalg.norm(np.array(center) - np.array(obj['center']))
                if dist < min_distance:
                    min_distance = dist
                    best_match = obj_id
            
            if best_match is not None and min_distance < 50:
                new_tracked_objects[best_match] = {
                    'bbox': (x1, y1, x2, y2),
                    'center': center,
                    'class': cls,
                    'age': 0
                }
            else:
                new_tracked_objects[self.next_id] = {
                    'bbox': (x1, y1, x2, y2),
                    'center': center,
                    'class': cls,
                    'age': 0
                }
                self.next_id += 1
        
        for obj_id in self.tracked_objects:
            if obj_id not in new_tracked_objects:
                self.tracked_objects[obj_id]['age'] += 1
                if self.tracked_objects[obj_id]['age'] < self.max_age:
                    new_tracked_objects[obj_id] = self.tracked_objects[obj_id]
        
        self.tracked_objects = new_tracked_objects
        return [(obj['bbox'][0], obj['bbox'][1], obj['bbox'][2], obj['bbox'][3], obj_id, obj['class']) 
                for obj_id, obj in self.tracked_objects.items()]

def main():
    video_path = r'C:/Users/DELL/Desktop/only traffic project/vehicle classification/traffic.mp4'
    monitor = ResolutionIndependentTrafficMonitor(video_path)
    monitor.process_video()

if __name__ == "__main__":
    main()



0: 384x640 4 persons, 9 cars, 3 motorcycles, 3 trucks, 33.3ms
Speed: 0.0ms preprocess, 33.3ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 9 cars, 3 motorcycles, 2 trucks, 20.9ms
Speed: 2.0ms preprocess, 20.9ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 9 cars, 3 motorcycles, 2 trucks, 27.4ms
Speed: 0.0ms preprocess, 27.4ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 persons, 9 cars, 3 motorcycles, 3 trucks, 14.5ms
Speed: 1.5ms preprocess, 14.5ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 9 cars, 3 motorcycles, 2 trucks, 17.7ms
Speed: 0.0ms preprocess, 17.7ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 9 cars, 3 motorcycles, 2 trucks, 14.5ms
Speed: 1.0ms preprocess, 14.5ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 9 cars, 3 motorcy

In [8]:
import cv2
import numpy as np
from ultralytics import YOLO
import cvzone
import matplotlib.pyplot as plt
import io

class_names = [
    'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 
    'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 
    'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 
    'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 
    'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 
    'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]

class_names_goal = ['car', 'motorcycle', 'bus', 'truck']

class ResolutionIndependentTrafficMonitor:
    def __init__(self, video_path, model_path=r'C:/Users/DELL/Desktop/only traffic project/yolov8/yolov8x.pt'):
        self.video = cv2.VideoCapture(video_path)
        if not self.video.isOpened():
            raise ValueError("Error opening video file")
        
        # Original video dimensions
        self.orig_width = int(self.video.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.orig_height = int(self.video.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.fps = self.video.get(cv2.CAP_PROP_FPS)
        
        # Calculate dimensions for grid layout
        self.grid_width = int(self.orig_width * 1.3)  # Total width including pie chart
        self.grid_height = self.orig_height
        self.video_width = int(self.grid_width * 0.7)  # 70% for video
        self.chart_width = self.grid_width - self.video_width  # 30% for chart
        
        # Initialize YOLOv8x model
        self.model = YOLO(model_path)
        
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        output_path = 'C:/Users/DELL/Desktop/only traffic project/vehicle classification/traffic_classify3.mp4'
        self.out = cv2.VideoWriter(output_path, fourcc, self.fps, (self.grid_width, self.grid_height))
        
        self.vehicle_count = {'left': 0, 'right': 0}
        self.counted_vehicle_ids = {'left': set(), 'right': set()}
        
        self.line_y = int(self.orig_height * 0.65)
        
        self.mask = self.create_mask()
        self.tracker = SimpleTracker()

    def create_pie_chart(self):
        plt.figure(figsize=(8, 8))
        plt.clf()
        
        # Enhanced color scheme for better visibility
        colors = ['#FF6B6B', '#4ECDC4']  # Coral Red and Turquoise
        lanes = list(self.vehicle_count.keys())
        total_counts = [self.vehicle_count[lane] for lane in lanes]
        
        total_vehicles = sum(total_counts)
        labels = [f'{lane.upper()}\n{count} vehicles' for lane, count in zip(lanes, total_counts)]
        sizes = total_counts if sum(total_counts) > 0 else [1, 1]
        
        plt.pie(
            sizes,
            labels=labels,
            colors=colors,
            autopct='%1.1f%%',
            startangle=90,
            wedgeprops={"linewidth": 2, "edgecolor": "white"},
            textprops={"fontsize": 12, "fontweight": "bold", "color": "white"}
        )
        
        plt.title(f'Vehicle Distribution\nTotal: {total_vehicles} vehicles',
                 color='white',
                 pad=20,
                 fontsize=14,
                 fontweight='bold')
        
        # Convert plot to image
        buf = io.BytesIO()
        plt.savefig(buf, format='png', facecolor='#1a1a1a', transparent=False, bbox_inches='tight', dpi=100)
        buf.seek(0)
        img_arr = np.frombuffer(buf.getvalue(), dtype=np.uint8)
        buf.close()
        plt.close()
        
        img = cv2.imdecode(img_arr, cv2.IMREAD_COLOR)
        return img

    def create_grid_layout(self, frame, pie_chart):
        # Create black canvas for grid
        grid = np.zeros((self.grid_height, self.grid_width, 3), dtype=np.uint8)
        
        # Resize video frame to fit 70% of grid
        video_frame = cv2.resize(frame, (self.video_width, self.grid_height))
        
        # Resize pie chart to fit 30% of grid
        pie_chart = cv2.resize(pie_chart, (self.chart_width, self.grid_height))
        
        # Place video and pie chart in grid
        grid[:, :self.video_width] = video_frame
        grid[:, self.video_width:] = pie_chart
        
        # Add separator line
        cv2.line(grid, 
                 (self.video_width, 0),
                 (self.video_width, self.grid_height),
                 (255, 255, 255),
                 2)
        
        return grid

    def create_mask(self):
        mask = np.zeros((self.orig_height, self.orig_width), dtype=np.uint8)
        pts = np.array([
            [int(self.orig_width * 0.15), int(self.orig_height * 0.55)],
            [int(self.orig_width * 0.85), int(self.orig_height * 0.55)],
            [self.orig_width, self.orig_height],
            [0, self.orig_height]
        ], np.int32)
        pts = pts.reshape((-1, 1, 2))
        cv2.fillPoly(mask, [pts], 255)
        return mask

    def detect_vehicles(self, image_region):
        detections = []
        results = self.model(image_region, stream=True)
        
        for r in results:
            for box in r.boxes:
                class_name = class_names[int(box.cls[0])]
                if class_name not in class_names_goal:
                    continue
                
                confidence = round(float(box.conf[0]) * 100, 2)
                if confidence < 60:  # Increased confidence threshold to reduce false positives
                    continue
                
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                detections.append([x1, y1, x2, y2, float(box.conf[0]), class_name])
        
        return detections

    def process_frame(self, frame):
        # Resize frame to original dimensions
        frame = cv2.resize(frame, (self.orig_width, self.orig_height))
        image_region = cv2.bitwise_and(frame, frame, mask=self.mask)
        
        # Detect and track vehicles
        detections = self.detect_vehicles(image_region)
        tracked_objects = self.tracker.update(detections)
        
        # Count vehicles and annotate frame
        self.count_vehicles(frame, tracked_objects)
        self.annotate_frame(frame, tracked_objects)
        
        # Create pie chart
        pie_chart = self.create_pie_chart()
        
        # Create grid layout
        grid_frame = self.create_grid_layout(frame, pie_chart)
        
        return grid_frame
    
    def count_vehicles(self, frame, tracked_objects):
        for obj in tracked_objects:
            x1, y1, x2, y2, obj_id, cls = obj
            center_x, center_y = (x1 + x2) // 2, (y1 + y2) // 2
            
            if self.line_y - 10 < center_y < self.line_y + 10:
                if center_x < self.orig_width // 2:  # Left side
                    if obj_id not in self.counted_vehicle_ids['left']:
                        self.counted_vehicle_ids['left'].add(obj_id)
                        self.vehicle_count['left'] += 1
                        cv2.line(frame, 
                                (0, self.line_y), 
                                (self.orig_width // 2, self.line_y), 
                                (46, 204, 113), 3)  # Enhanced green color
                else:  # Right side
                    if obj_id not in self.counted_vehicle_ids['right']:
                        self.counted_vehicle_ids['right'].add(obj_id)
                        self.vehicle_count['right'] += 1
                        cv2.line(frame, 
                                (self.orig_width // 2, self.line_y), 
                                (self.orig_width, self.line_y), 
                                (46, 204, 113), 3)  # Enhanced green color
    
    def annotate_frame(self, frame, tracked_objects):
        # Draw detection line
        cv2.line(frame, (0, self.line_y), (self.orig_width, self.line_y),
                 (255, 255, 255), 1, cv2.LINE_AA)
        
        for obj in tracked_objects:
            x1, y1, x2, y2, obj_id, cls = obj
            # Create background box for text
            cvzone.cornerRect(frame, (x1, y1, x2-x1, y2-y1), l=9, rt=2, colorR=(255, 255, 255))
            
            # Enhanced text display
            text = f'ID: {obj_id} {cls}'
            font_scale = 0.6
            thickness = 2
            font = cv2.FONT_HERSHEY_SIMPLEX
            
            # Get text size
            (text_width, text_height), _ = cv2.getTextSize(text, font, font_scale, thickness)
            
            # Draw text background
            cv2.rectangle(frame,
                         (x1, y1 - text_height - 10),
                         (x1 + text_width + 10, y1),
                         (46, 204, 113),
                         -1)
            
            # Draw text
            cv2.putText(frame,
                       text,
                       (x1 + 5, y1 - 5),
                       font,
                       font_scale,
                       (255, 255, 255),
                       thickness)
        
        # Add lane counts with enhanced styling
        cvzone.putTextRect(frame,
                          f'Left Lane: {self.vehicle_count["left"]}',
                          (20, 40),
                          scale=2,
                          thickness=3,
                          offset=10,
                          colorR=(52, 73, 94),
                          colorT=(255, 255, 255))
        
        cvzone.putTextRect(frame,
                          f'Right Lane: {self.vehicle_count["right"]}',
                          (self.orig_width - 280, 40),
                          scale=2,
                          thickness=3,
                          offset=10,
                          colorR=(52, 73, 94),
                          colorT=(255, 255, 255))

    def process_video(self):
        while True:
            success, frame = self.video.read()
            if not success:
                break
            
            processed_frame = self.process_frame(frame)
            
            # Display video with grid layout
            cv2.imshow('Traffic Monitoring', processed_frame)
            self.out.write(processed_frame)
            
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        
        self.video.release()
        self.out.release()
        cv2.destroyAllWindows()

class SimpleTracker:
    def __init__(self, max_age=20):
        self.next_id = 1
        self.tracked_objects = {}
        self.max_age = max_age

    def update(self, detections):
        new_tracked_objects = {}
        
        for det in detections:
            x1, y1, x2, y2, conf, cls = det
            center = ((x1 + x2) // 2, (y1 + y2) // 2)
            
            best_match = None
            min_distance = float('inf')
            
            for obj_id, obj in self.tracked_objects.items():
                dist = np.linalg.norm(np.array(center) - np.array(obj['center']))
                if dist < min_distance:
                    min_distance = dist
                    best_match = obj_id
            
            if best_match is not None and min_distance < 50:
                new_tracked_objects[best_match] = {
                    'bbox': (x1, y1, x2, y2),
                    'center': center,
                    'class': cls,
                    'age': 0
                }
            else:
                new_tracked_objects[self.next_id] = {
                    'bbox': (x1, y1, x2, y2),
                    'center': center,
                    'class': cls,
                    'age': 0
                }
                self.next_id += 1
        
        for obj_id in self.tracked_objects:
            if obj_id not in new_tracked_objects:
                self.tracked_objects[obj_id]['age'] += 1
                if self.tracked_objects[obj_id]['age'] < self.max_age:
                    new_tracked_objects[obj_id] = self.tracked_objects[obj_id]
        
        self.tracked_objects = new_tracked_objects
        return [(obj['bbox'][0], obj['bbox'][1], obj['bbox'][2], obj['bbox'][3], obj_id, obj['class']) 
                for obj_id, obj in self.tracked_objects.items()]

def main():
    video_path = r'C:/Users/DELL/Desktop/only traffic project/vehicle classification/traffic.mp4'
    monitor = ResolutionIndependentTrafficMonitor(video_path)
    monitor.process_video()

if __name__ == "__main__":
    main()


0: 384x640 4 persons, 9 cars, 3 motorcycles, 3 trucks, 33.3ms
Speed: 0.0ms preprocess, 33.3ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 9 cars, 3 motorcycles, 2 trucks, 35.3ms
Speed: 0.0ms preprocess, 35.3ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 9 cars, 3 motorcycles, 2 trucks, 25.7ms
Speed: 1.0ms preprocess, 25.7ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 persons, 9 cars, 3 motorcycles, 3 trucks, 30.0ms
Speed: 0.0ms preprocess, 30.0ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 9 cars, 3 motorcycles, 2 trucks, 29.9ms
Speed: 0.0ms preprocess, 29.9ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 9 cars, 3 motorcycles, 2 trucks, 29.7ms
Speed: 0.0ms preprocess, 29.7ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 9 cars, 3 motorcy